# Feature Engineering, Model Training & Selection
This notebook demonstrates feature engineering, training multiple machine learning estimators, comparing CV scores, and saving the final prediction pipeline.

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split

## 1. Load Clean Data and Split
We'll load the data and split it into training and validation sets.

In [ ]:
import sys
sys.path.append('../')
from src.data_loader import load_data
from src.preprocessing import AmesHousingPreprocessor, remove_outliers

train_df, test_df = load_data(data_dir="../dataset")
train_df = remove_outliers(train_df)

X = train_df.drop(columns=['SalePrice'])
y = np.log1p(train_df['SalePrice'])

X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)
print(f"Train size: {X_train.shape[0]} | Val size: {X_val.shape[0]}")

## 2. Apply Preprocessing and Feature Engineering
We preprocess the raw inputs, apply our custom feature engineering, and scale our numerical features.

In [ ]:
from src.feature_engineering import AmesHousingFeatureEngineer

# Preprocess
preprocessor = AmesHousingPreprocessor()
preprocessor.fit(X_train)
X_train_pre = preprocessor.transform(X_train)
X_val_pre = preprocessor.transform(X_val)

# Feature Engineer & Scale
engineer = AmesHousingFeatureEngineer(scale_numerical=True)
engineer.fit(X_train_pre)
X_train_eng = engineer.transform(X_train_pre)
X_val_eng = engineer.transform(X_val_pre)

print("Engineered Train shape:", X_train_eng.shape)
print("Engineered Val shape:", X_val_eng.shape)

## 3. Train and Compare Models
We train six different models (Linear Regression, Decision Tree, Random Forest, Gradient Boosting, XGBoost, and LightGBM) and evaluate them using 5-fold cross-validation.

In [ ]:
from src.model import train_and_evaluate_models, get_best_model

results = train_and_evaluate_models(X_train_eng, y_train, cv_folds=5)

In [ ]:
# Compare models visually
from src.evaluate import plot_model_comparison
plot_model_comparison(results, save_dir="../outputs/graphs")

# Display model scores
for name, info in results.items():
    print(f"{name:20} Mean CV RMSE (Log Scale): {info['cv_rmse_mean']:.4f}")

## 4. Evaluate the Best Model on Validation Set
Let's select the best model, predict on the local validation set, evaluate key regression metrics, and produce diagnostic plots.

In [ ]:
best_name, best_model = get_best_model(results)

from src.evaluate import calculate_metrics, plot_predictions_vs_actual, plot_residuals, plot_feature_importance, plot_learning_curve

y_val_pred = best_model.predict(X_val_eng)
metrics = calculate_metrics(y_val, y_val_pred, is_log=True)

print(f"Best Model: {best_name}")
for k, v in metrics.items():
    print(f"{k:10} : {v:.4f}")

In [ ]:
# Generate graphs
plot_predictions_vs_actual(y_val, y_val_pred, save_dir="../outputs/graphs")
plot_residuals(y_val, y_val_pred, save_dir="../outputs/graphs")
feature_names = [col for col in X_train_eng.columns if col != 'Id']
plot_feature_importance(best_model, feature_names, save_dir="../outputs/graphs")
plot_learning_curve(best_model, X_train_eng, y_train, save_dir="../outputs/graphs")

## 5. Retrain Pipeline on Full Dataset & Save
We train the best selected estimator on the complete `train.csv` dataset and save the entire pipeline (model + preprocessor + feature_engineer) to a single Pickle file.

In [ ]:
from src.model import save_model

# Fit full pipeline preprocessors
full_pre = AmesHousingPreprocessor()
full_pre.fit(X)
X_full_pre = full_pre.transform(X)

full_eng = AmesHousingFeatureEngineer(scale_numerical=True)
full_eng.fit(X_full_pre)
X_full_eng = full_eng.transform(X_full_pre)

# Fit model on full training set
best_model.fit(X_full_eng, y)

# Save
os.makedirs("../models", exist_ok=True)
save_model(best_model, full_pre, full_eng, "../models/trained_model.pkl")